# 08: Anomalies Over Time

Answers a question `05` and `06` each deferred to the other: does a locality-year flagged anomalous by `05`'s Isolation Forest tend to belong to a locality that's also volatile over time in `06`'s sense (cluster membership flipping between elections)? No new model here -- joins `locality_anomalies.parquet` (`05`) and `locality_cluster_transitions.parquet` (`06`) and checks the relationship directly.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, str(Path("..") / "src"))

%matplotlib inline
pd.set_option("display.max_columns", 40)
print("pandas:", pd.__version__)

pandas: 3.0.2


In [2]:
CONFIG = {
    "processed_dir": "../data/processed",
}

processed_dir = Path(CONFIG["processed_dir"])
print("Config set.")

Config set.


## Load `05`'s anomaly flags and `06`'s transitions

In [3]:
anomalies = pd.read_parquet(processed_dir / "locality_anomalies.parquet")
transitions = pd.read_parquet(processed_dir / "locality_cluster_transitions.parquet")

print("Anomalies (locality x year):", anomalies.shape, "-- years:", sorted(anomalies["year"].unique()))
print(f"Overall outlier rate: {anomalies['is_outlier'].mean():.1%} (fixed at ~2% by `05`'s contamination setting)")
print("\nTransitions (locality x consecutive election pair):", transitions.shape)

Anomalies (locality x year): (8967, 14) -- years: [np.int64(2010), np.int64(2013), np.int64(2016), np.int64(2019), np.int64(2022), np.int64(2025)]
Overall outlier rate: 2.0% (fixed at ~2% by `05`'s contamination setting)

Transitions (locality x consecutive election pair): (7187, 7)


## Per-locality summaries

Collapses both files from locality-year/locality-transition grain to one row per locality: years flagged anomalous, and how often its cluster label changed between consecutive elections.

In [4]:
loc_anomaly = anomalies.groupby(["province", "city"]).agg(
    n_years_seen=("is_outlier", "size"),
    n_years_flagged=("is_outlier", "sum"),
)
loc_anomaly["ever_flagged"] = loc_anomaly["n_years_flagged"] > 0

loc_volatility = transitions.groupby(["province", "city"]).agg(
    n_transitions=("changed", "size"),
    n_changes=("changed", "sum"),
)
loc_volatility["change_rate"] = loc_volatility["n_changes"] / loc_volatility["n_transitions"]

print(f"{len(loc_anomaly):,} localities have an anomaly summary "
      f"({loc_anomaly['ever_flagged'].sum():,} were flagged anomalous at least once).")
print(f"{len(loc_volatility):,} localities have a volatility summary "
      f"(need 2+ observed years, per `06`).")

1,780 localities have an anomaly summary (144 were flagged anomalous at least once).
1,720 localities have a volatility summary (need 2+ observed years, per `06`).


## Do ever-flagged localities have a higher change rate?

Hypothesis: a locality with one wildly unusual election (`05`) might also be one whose vote-shape cluster flips frequently (`06`) -- both reflecting the same underlying instability. Inner join on `(province, city)`, split by whether a locality was ever flagged, compare change rates with a two-sided Mann-Whitney U test (chosen over a t-test since change rate is a bounded proportion on discrete transition counts, not continuous/normal).

In [5]:
merged = loc_anomaly.join(loc_volatility, how="inner")
print(f"{len(merged):,} localities present in both summaries.")

summary_by_flag = merged.groupby("ever_flagged")["change_rate"].agg(["mean", "median", "count"])
print("\nChange rate by whether the locality was ever flagged anomalous:")
display(summary_by_flag.round(3))

flagged_rates = merged.loc[merged["ever_flagged"], "change_rate"]
unflagged_rates = merged.loc[~merged["ever_flagged"], "change_rate"]
u_stat, p_value = stats.mannwhitneyu(flagged_rates, unflagged_rates, alternative="two-sided")
print(f"\nMann-Whitney U test (flagged vs. never-flagged change rate): "
      f"U={u_stat:,.0f}, p={p_value:.3f}")
if p_value < 0.05:
    direction = "higher" if flagged_rates.mean() > unflagged_rates.mean() else "lower"
    print(f"-> Statistically significant at p<0.05: ever-flagged localities have a {direction} change rate.")
else:
    print("-> Not statistically significant at p<0.05 -- no reliable difference detected.")

1,720 localities present in both summaries.

Change rate by whether the locality was ever flagged anomalous:


,mean,median,count
ever_flagged,,,
False,0.290,0.25,1585
True,0.241,0.20,135



Mann-Whitney U test (flagged vs. never-flagged change rate): U=96,716, p=0.056
-> Not statistically significant at p<0.05 -- no reliable difference detected.


## A finer-grained check: is a specific transition more likely to be a *change* when one of its two endpoint years was itself flagged anomalous?

The locality-level test above averages away when a locality was flagged. This checks the transition level directly: for a given election-to-election transition, was either endpoint year flagged by `05`, and if so, is that transition more likely to be a cluster change than one with no flagged endpoint?

In [6]:
anomaly_lookup = anomalies.set_index(["province", "city", "year"])["is_outlier"]

transitions = transitions.copy()
from_key = pd.MultiIndex.from_frame(transitions[["province", "city", "year_from"]].rename(columns={"year_from": "year"}))
to_key = pd.MultiIndex.from_frame(transitions[["province", "city", "year_to"]].rename(columns={"year_to": "year"}))
transitions["from_anomalous"] = anomaly_lookup.reindex(from_key).fillna(False).to_numpy()
transitions["to_anomalous"] = anomaly_lookup.reindex(to_key).fillna(False).to_numpy()
transitions["either_anomalous"] = transitions["from_anomalous"] | transitions["to_anomalous"]

by_either = transitions.groupby("either_anomalous")["changed"].agg(["mean", "count"])
print("Fraction of transitions that were a cluster change, by whether either endpoint year was flagged anomalous:")
display(by_either.round(3))

chi2_table = pd.crosstab(transitions["either_anomalous"], transitions["changed"])
chi2, chi2_p, _, _ = stats.chi2_contingency(chi2_table)
print(f"\nChi-square test of independence: chi2={chi2:.2f}, p={chi2_p:.3f}")

Fraction of transitions that were a cluster change, by whether either endpoint year was flagged anomalous:


,mean,count
either_anomalous,,
False,0.284,6938
True,0.277,249



Chi-square test of independence: chi2=0.02, p=0.882


## Spot check: a persistently-flagged locality, looked at directly

The locality flagged anomalous in the most years, with its full cluster history and anomaly scores side by side -- if repeated unusualness doesn't translate into instability, this locality should keep getting flagged while its cluster label barely moves.

In [7]:
most_flagged_idx = loc_anomaly["n_years_flagged"].idxmax()
most_flagged_province, most_flagged_city = most_flagged_idx
print(f"Most-often-flagged locality: {most_flagged_city}, {most_flagged_province} "
      f"({loc_anomaly.loc[most_flagged_idx, 'n_years_flagged']} of "
      f"{loc_anomaly.loc[most_flagged_idx, 'n_years_seen']} observed years flagged)")

spot = anomalies[(anomalies["province"] == most_flagged_province) & (anomalies["city"] == most_flagged_city)] \
    .sort_values("year")[["year", "is_outlier", "anomaly_score", "MAYOR_margin", "MAYOR_enc"]]
print("\nAnomaly flags across every observed year:")
display(spot)

clusters_here = pd.read_parquet(processed_dir / "locality_clusters.parquet")
clusters_here = clusters_here[(clusters_here["province"] == most_flagged_province)
                               & (clusters_here["city"] == most_flagged_city)] \
    .sort_values("year")[["year", "cluster"]]
print("\nCluster label (from `04`) across every observed year:")
display(clusters_here)

if most_flagged_idx in loc_volatility.index:
    print(f"\nChange rate for this locality: {loc_volatility.loc[most_flagged_idx, 'change_rate']:.1%} "
          f"({loc_volatility.loc[most_flagged_idx, 'n_changes']} changes of "
          f"{loc_volatility.loc[most_flagged_idx, 'n_transitions']} transitions) "
          f"-- vs. {merged['change_rate'].mean():.1%} average across all localities.")

Most-often-flagged locality: DAVAO, DAVAO DEL SUR (4 of 5 observed years flagged)



Anomaly flags across every observed year:


,year,is_outlier,anomaly_score,MAYOR_margin,MAYOR_enc
1416,2013,False,0.009526,1.000000,1.000000
2967,2016,True,-0.001375,0.992027,1.009042
4600,2019,True,-0.006194,0.985394,1.014712
6227,2022,True,-0.017768,0.791983,1.238184
7861,2025,True,-0.024291,0.771866,1.274701



Cluster label (from `04`) across every observed year:


,year,cluster
1416,2013,0
2967,2016,0
4600,2019,0
6227,2022,0
7861,2025,1



Change rate for this locality: 25.0% (1 changes of 4 transitions) -- vs. 28.6% average across all localities.


## Sanity checks

In [8]:
assert merged["change_rate"].between(0, 1).all(), "change_rate out of [0,1] range"
assert (loc_anomaly["n_years_flagged"] <= loc_anomaly["n_years_seen"]).all(), "flagged years exceed observed years somewhere"
assert len(merged) > 1000, f"Merged sample unexpectedly small: {len(merged)}"
print(f"All sanity checks passed on {len(merged):,} localities.")

All sanity checks passed on 1,720 localities.


## Save

In [9]:
anomaly_temporal = merged.reset_index()[
    ["province", "city", "n_years_seen", "n_years_flagged", "ever_flagged",
     "n_transitions", "n_changes", "change_rate"]
]
anomaly_temporal.to_parquet(processed_dir / "locality_anomaly_temporal.parquet", index=False)
print("Saved locality_anomaly_temporal.parquet:", anomaly_temporal.shape)

Saved locality_anomaly_temporal.parquet: (1720, 8)


## Summary

Tested whether a locality having an unusual vote-shape in a single election (`05`) predicts whether its vote-shape category shifts across elections (`06`). Both the locality-level test (ever-flagged vs. never-flagged change rate, Mann-Whitney p=0.056) and the transition-level test (chi-square p=0.882) come back non-significant. An anomalous election, in this data, looks like a one-off departure in that race's numbers, not a symptom of a generally unstable locality.

**Limits of this null result:** `05`'s contamination setting fixes the flagged rate at ~2% of locality-years by construction, so the flagged-vs-unflagged comparison has limited statistical power. "No significant difference detected" is not the same as "no relationship exists."

**Out of scope here:** re-running `05`'s Isolation Forest at a different contamination level for more power, and any geographic pattern in where anomaly and volatility do or don't coincide.